In [2]:

import re

# Three synthetic documents
documents = {
    "DOC1": """University assignments must be submitted before the announced deadline. Students should upload their work through the official learning management system. Late submissions may receive reduced marks according to university policy. However, students who face a genuine medical emergency can request an extension. They must inform the instructor before the deadline whenever possible. A medical certificate or other relevant evidence should be provided with the request. The instructor will review the application and decide whether an extension can be granted. Approval is not automatic, and students should wait for confirmation before submitting late work. Students should keep a copy of their submitted assignment and confirmation message. If the LMS is unavailable, they should take a screenshot and inform the instructor immediately. Technical problems should be reported with the date and time of the issue. Students are responsible for checking announcements and following updated instructions. Assignments sent through unofficial channels may not be accepted. The purpose of these rules is to ensure fair treatment for all students.""",

    "DOC2": """Customers should report a damaged parcel within 48 hours of delivery. They should contact the support team and provide clear photographs of the damaged item and its packaging. The order number and delivery date should also be included in the complaint. The support team will review the information and contact the customer if more details are needed. A refund or replacement may be offered after the complaint is checked. However, a refund is not guaranteed in every case. The decision depends on the investigation and the condition of the parcel. Customers should keep the original packaging until the complaint has been resolved. Claims submitted after the 48-hour period may require additional review. If photographs are missing, the support team may ask the customer to send them before processing the request. Customers should use the official support email or help centre to submit complaints. Messages sent to unofficial social media accounts may not be treated as formal requests. The support team normally provides an update after reviewing the available evidence. Customers should include accurate contact information so that the team can respond without delay.""",

    "DOC3": """The software system uses an access token to authenticate requests sent to the API. Each token has an expiry time and cannot be used after it expires. When a token expires, the client should request a new token by using the refresh token. The refresh request must be sent to the authentication endpoint. The client should store the new access token securely and use it for future requests. If the refresh token is invalid or expired, the user may need to sign in again. Developers should check the response status before retrying a failed request. Repeated requests with an expired token may result in authentication errors. The API documentation should be checked whenever an endpoint or authentication rule changes. Developers should also record the API version used by their application. The system must not expose access tokens in public logs, screenshots, or error messages. A developer should test token renewal before releasing an application to users, because an incorrect refresh process can interrupt several connected services and cause requests to fail even when the user has entered the correct login details. The client must handle these errors safely and guide the user when another sign-in is required."""
}

# Evidence phrases
evidence = {
    "DOC1": [
        "Late submissions may receive reduced marks",
        "A medical certificate or other relevant evidence"
    ],
    "DOC2": [
        "within 48 hours of delivery",
        "a refund is not guaranteed in every case"
    ],
    "DOC3": [
        "When a token expires",
        "If the refresh token is invalid or expired"
    ]
}

# Count words
def word_count(text):
    return len(text.split())


# Fixed-size word windows
def windows(text, size=40, overlap=0, offset=0):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Size must be positive and overlap must be smaller than size.")

    words = list(re.finditer(r"\S+", text))
    chunks = []
    step = size - overlap

    for i in range(0, len(words), step):
        start = words[i].start()
        end = words[min(i + size, len(words)) - 1].end()

        chunks.append({
            "text": text[start:end],
            "start": offset + start,
            "end": offset + end
        })

    return chunks


# Split text into sentences
def split_sentences(text):
    sentences = []
    start = 0

    for match in re.finditer(r"(?<=[.!?])\s+", text):
        sentence = text[start:match.start()].strip()

        if sentence:
            left = len(text[start:match.start()]) - len(text[start:match.start()].lstrip())
            right = len(text[start:match.start()].rstrip())
            sentences.append((start + left, start + right))

        start = match.end()

    sentence = text[start:].strip()

    if sentence:
        left = len(text[start:]) - len(text[start:].lstrip())
        right = len(text[start:].rstrip())
        sentences.append((start + left, start + right))

    return sentences


# Sentence-aware chunks
def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("Size must be positive.")

    sentences = split_sentences(text)
    chunks = []
    current_start = None
    current_end = None
    current_words = 0

    for start, end in sentences:
        sentence = text[start:end]
        count = word_count(sentence)

        if count > size:
            if current_start is not None:
                chunks.append({
                    "text": text[current_start:current_end],
                    "start": current_start,
                    "end": current_end
                })

                current_start = None
                current_end = None
                current_words = 0

            long_chunks = windows(sentence, size, 0, offset=start)
            chunks.extend(long_chunks)

        elif current_words + count <= size:
            if current_start is None:
                current_start = start

            current_end = end
            current_words += count

        else:
            chunks.append({
                "text": text[current_start:current_end],
                "start": current_start,
                "end": current_end
            })

            current_start = start
            current_end = end
            current_words = count

    if current_start is not None:
        chunks.append({
            "text": text[current_start:current_end],
            "start": current_start,
            "end": current_end
        })

    return chunks


# Add metadata
def add_metadata(doc_id, source_path, chunks, source_text):
    result = []

    for i, chunk in enumerate(chunks, 1):
        result.append({
            "document_id": doc_id,
            "chunk_id": f"{doc_id}_CH{i}",
            "source_path": source_path,
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"]
        })

        assert chunk["text"] == source_text[chunk["start"]:chunk["end"]]

    return result


# Compare chunking methods
def compare_chunks(doc_id, source_text, chunks):
    original_words = word_count(source_text)
    total_words = sum(word_count(chunk["text"]) for chunk in chunks)
    duplicate_words = total_words - original_words

    sizes = [word_count(chunk["text"]) for chunk in chunks]
    survived = 0

    for phrase in evidence[doc_id]:
        found = any(
            phrase.lower() in chunk["text"].lower()
            for chunk in chunks
        )

        if found:
            survived += 1

    print("Number of chunks:", len(chunks))
    print("Chunk sizes:", sizes)
    print("Duplicated words:", duplicate_words)
    print("Evidence phrases preserved:", survived, "/", len(evidence[doc_id]))


# Run all chunking methods
for doc_id, text in documents.items():

    source_path = "synthetic/" + doc_id + ".txt"

    print("\n" + "=" * 50)
    print("DOCUMENT:", doc_id)
    print("Original word count:", word_count(text))

    print("\n1. Fixed Windows - Overlap 0")
    chunks_0 = windows(text, 40, 0)
    result_0 = add_metadata(doc_id, source_path, chunks_0, text)

    for chunk in result_0:
        print(chunk)

    compare_chunks(doc_id, text, chunks_0)

    print("\n2. Fixed Windows - Overlap 10")
    chunks_10 = windows(text, 40, 10)
    result_10 = add_metadata(doc_id, source_path, chunks_10, text)

    for chunk in result_10:
        print(chunk)

    compare_chunks(doc_id, text, chunks_10)

    print("\n3. Sentence-Aware Chunks")
    chunks_sentence = sentence_chunks(text, 40)
    result_sentence = add_metadata(
        doc_id, source_path, chunks_sentence, text
    )

    for chunk in result_sentence:
        print(chunk)

    compare_chunks(doc_id, text, chunks_sentence)


# Empty text test
print("\nEMPTY TEXT TEST")
print("Fixed windows:", windows("", 40, 0))
print("Sentence chunks:", sentence_chunks("", 40))


# Short text test
print("\nSHORT TEXT TEST")
short_text = "This is a short example."
print("Fixed windows:", windows(short_text, 40, 0))
print("Sentence chunks:", sentence_chunks(short_text, 40))


# Invalid overlap test
print("\nINVALID OVERLAP TEST")
try:
    windows("This is a test.", 40, 40)
except ValueError as error:
    print(error)


# Long sentence test
print("\nLONG SENTENCE TEST")
long_sentence = (
    "Students should carefully read all instructions before submitting "
    "their assignments because missing information can create problems "
    "during checking and may require them to contact the instructor "
    "for clarification before the final submission deadline."
)

long_chunks = sentence_chunks(long_sentence, 10)

for chunk in long_chunks:
    print(chunk, "| Words:", word_count(chunk["text"]))


# Abbreviation test
print("\nABBREVIATION TEST")
abbreviation_text = "Dr. Khan explained the topic. Students asked questions."
print("Detected sentences:", split_sentences(abbreviation_text))



DOCUMENT: DOC1
Original word count: 165

1. Fixed Windows - Overlap 0
{'document_id': 'DOC1', 'chunk_id': 'DOC1_CH1', 'source_path': 'synthetic/DOC1.txt', 'start': 0, 'end': 296, 'text': 'University assignments must be submitted before the announced deadline. Students should upload their work through the official learning management system. Late submissions may receive reduced marks according to university policy. However, students who face a genuine medical emergency can request'}
{'document_id': 'DOC1', 'chunk_id': 'DOC1_CH2', 'source_path': 'synthetic/DOC1.txt', 'start': 297, 'end': 567, 'text': 'an extension. They must inform the instructor before the deadline whenever possible. A medical certificate or other relevant evidence should be provided with the request. The instructor will review the application and decide whether an extension can be granted. Approval'}
{'document_id': 'DOC1', 'chunk_id': 'DOC1_CH3', 'source_path': 'synthetic/DOC1.txt', 'start': 568, 'end': 836, 'text': 